In [1]:
import os
import torch
import json
import requests
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

In [2]:
def get_all_image_paths(folder_path, extensions={'.jpg', '.jpeg', '.png', '.bmp', '.gif'}):
    image_paths = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if os.path.splitext(file)[1].lower() in extensions:
                image_paths.append(os.path.join(root, file))

    print( f"[INFO] Found {len(image_paths)} images in {folder_path}")
        
    return image_paths

# ------------------------------
# 1. Load YOLOv12 Model (Adjust if needed)
# ------------------------------
def load_model(weights_path):
    # model = torch.hub.load('ultralytics/yolov5', 'custom', path=weights_path)
    model = YOLO(weights_path)
    return model

# ------------------------------
# 2. Run Inference and Convert to COCO
# ------------------------------
def run_inference(model, image_dir, output_dir, categories=[]):
    os.makedirs(output_dir, exist_ok=True)

    print("[INFO] Running inference on images in:", image_dir)

    ann_id = 1
    category_set = set()
    image_files = get_all_image_paths(image_dir) #list(Path(image_dir).glob(f"*.{exts}"))

    print(image_files)

    coco_annotations = {
        "categories": categories,
        "images": [],
        "annotations": []
    }

    for img_id, image_path in enumerate(image_files, start=1):
        image = Image.open(image_path)
        width, height = image.size
        results = model(str(image_path))

        coco_annotations["images"].append({
            "id": img_id,
            "file_name": os.path.basename(image_path),
            "width": width,
            "height": height
        })

        for det in results[0].boxes:
            x1, y1, x2, y2 = det.xyxy[0].tolist()
            # conf = det.conf[0].item()
            class_id = int(det.cls.tolist()[0])

            w, h = x2 - x1, y2 - y1
            category_set.add(int(class_id))

            coco_annotations["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": int(class_id) + 1,  # COCO categories start from 1
                "bbox": [x1, y1, w, h],
                "area": w * h,
                "iscrowd": 0
            })
            ann_id += 1

    # for class_id in sorted(category_set):
        # coco_annotations["categories"].append({
        #     "id": class_id,
        #     "name": str(class_id),
        #     "supercategory": "none"
        # })

    output_json = os.path.join(output_dir, "annotations.json")
    with open(output_json, "w") as f:
        json.dump(coco_annotations, f, indent=4)

    print(f"[INFO] COCO annotations saved at: {output_json}")
    return output_json

In [3]:
# ------------------------------
# 3. Upload to CVAT
# ------------------------------
def update_cvat_annotations(
    cvat_host: str,
    username: str,
    password: str,
    task_id: int,
    coco_json_path: str,
    format: str = "COCO 1.0"
):
    session = requests.Session()

    # Authenticate
    login_data = {"username": username, "password": password}
    login_response = session.post(f"{cvat_host}/api/auth/login", json=login_data)
    token = login_response.json().get("key")
    if login_response.status_code != 200:
        raise Exception(f"Login failed: {login_response.text}")
    print("[INFO] Logged in successfully")

    # Upload annotations (replace mode)
    with open(coco_json_path, "rb") as f:
        headers = {
            "Authorization": f"Token {token}"
        }
        files = {"annotation_file": ("annotations.json", f, "application/json")}
        params = {
            "format": format,
            "loader": format,
            "job": "",
            "action": "upload",
            "update": "replace"  # Ensure it overwrites old annotations
        }
        response = session.put(
            f"{cvat_host}/api/tasks/{task_id}/annotations",
            headers=headers,
            files=files,
            params=params
        )
    if response.status_code == 202:
        print(f"[INFO] Uploaded and replacing annotations in task {task_id} successfully.")
    else:
        raise Exception(f"Annotation update failed: {response.text}")

In [5]:
# ------------------------------
# 4. Main Entry Point
# ------------------------------
if __name__ == "__main__":
    # /home/null/ultralytics/runs/detect/train65/weights/best_yolov12n_train65_adapter_v1.4.pt
    train_number = 83  # Adjust as needed
    root_weight_path = f"/home/null/ultralytics/runs/detect/train{train_number}/weights"  # Adjust path as needed
    weights_path = f"D:/Charger_Defect_Detection_Realtime/weights/best_yolov12n_train110_scratch_v1.6.pt"             # Path to your trained model
    
    root_data = "D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4"  # Adjust path as needed
    image_dir = f"{root_data}/images"          # Folder with images
    output_dir = f"{root_data}/images"                   # Folder to store JSON output
    
    # Define categories for COCO format
    categories = [
        {
            "id": 1,
            "name": "Defect",
            "supercategory": "none"
        },
        {
            "id": 2,
            "name": "Text_Defect",
            "supercategory": "none"
        },
        {
            "id": 3,
            "name": "QR_Code",
            "supercategory": "none"
        },
        {
            "id": 4,
            "name": "QR_Code_Defect",
            "supercategory": "none"
        },
        {
            "id": 5,
            "name": "Serial_Number",
            "supercategory": "none"
        },
    ]
    
    coco_json_path = run_inference(load_model(weights_path), image_dir, output_dir, categories)

    print(f"[INFO] COCO annotations generated at: {coco_json_path}")

    # Upload to CVAT
    cvat_host = "http://192.168.50.159:8085"     # Change to your CVAT server IP/port
    username = "null"
    password = "hieunt@2025"
    task_id = 63  # your existing CVAT task

    update_cvat_annotations(cvat_host, username, password, task_id, coco_json_path)

[INFO] Running inference on images in: D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4/images
[INFO] Found 73 images in D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4/images
['D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\1.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\2.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\3.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\4.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\5.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\5000_1.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\5000_2.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\6000_1.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\6000_2.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T2510_v0.1.4/images\\7000_1.jpg', 'D:\\CHARGER\\Abumentation\\Charger_v2.0\\EP-T251

<>:10: SyntaxWarning: invalid escape sequence '\C'
<>:10: SyntaxWarning: invalid escape sequence '\C'
C:\Users\Engineer\AppData\Local\Temp\ipykernel_42848\2711783369.py:10: SyntaxWarning: invalid escape sequence '\C'
  root_data = "D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4"  # Adjust path as needed


image 1/1 D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4\images\2.jpg: 640x608 2 Text_Defects, 1 QR_Code, 1 Serial_Number, 52.0ms
Speed: 2.1ms preprocess, 52.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 608)

image 1/1 D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4\images\3.jpg: 640x608 1 QR_Code, 1 Serial_Number, 51.7ms
Speed: 2.0ms preprocess, 51.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 608)

image 1/1 D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4\images\4.jpg: 640x608 1 QR_Code, 1 Serial_Number, 60.6ms
Speed: 2.3ms preprocess, 60.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 608)

image 1/1 D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4\images\5.jpg: 640x608 1 QR_Code, 1 Serial_Number, 54.7ms
Speed: 2.1ms preprocess, 54.7ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 608)

image 1/1 D:\CHARGER\Abumentation\Charger_v2.0\EP-T2510_v0.1.4\images\5000_1.jpg: 640x608 1 QR_Code, 1 Serial_Nu